# Drum: seluruh lagu dalam satu panggilan

**Hasil uji full-song ini telah ditolak pendengar:** groove, timbre dan timing tidak jelas. Baseline 30 detik tetap disimpan. Notebook ini dipertahankan untuk reproduksi; default DRY_RUN=True agar Run all tidak mengulang inferensi mahal. Ubah False hanya untuk eksperimen reproduksi yang disengaja, bukan sebagai perbaikan yang sudah terbukti.


Jalankan pada **VM Colab yang runtime dan bobot ACE-Step-nya masih tersedia**. Salin sel notebook ini ke sesi aktif jika perlu. Helper sudah disertakan, sehingga tidak perlu ZIP tools. Piano asli dipakai kembali jika SOURCE_PATH masih tersedia; kalau tidak, upload piano asli.

Ini uji baru: satu konteks penuh 10–600 detik, tanpa chunk musik independen atau stitching. Untuk piano 4:03, model menerima 244 detik dengan padding akhir kurang dari satu detik, lalu padding dibuang dari ekspor. Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah / caption drum tetap sama. Bobot/runtime dipakai kembali, tidak dipasang ulang.

Hasil START=90 dilaporkan banyak glitch; START=210 dinilai tanpa glitch. Keduanya memakai setting yang sama. Kualitas lagu penuh belum diketahui dan kebutuhan VRAM bisa berbeda. Jika OOM atau output gagal validasi, proses menyimpan report; tidak mengganti mode menjadi chunk atau mengganti precision. Baseline 30 detik dan hasil bass tetap disimpan.

DRY_RUN=True hanya menyiapkan plan. VM kosong perlu sel setup notebook 12 sampai bobot siap dahulu, tanpa menjalankan generasi bassnya.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid
from google.colab import files
DRY_RUN = True
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "drum_probe.py": {
    "sha256": "e1197fddd057474b7cc0b9258bb0851a15996fd9e2d96281a99bd4fdf65dc50c",
    "gzip_base64": "H4sIAAAAAAAC/7Va3XPbNhJ/11+B5B5AthRlx9NmTj4+uI1z07lem0vSduY0Gg5MghJOFMkjQNlOxv/77eKD30rSds4PsQkuFov9/O0yz58/f1U3R1LV5R2XpKxJ1uT5UpbFjkhR7HK+rJiUpMEHovacVKIoeEpuvr9dvlO8It//8uqG1E2hxJGHz58/X4hjVdaKsHpXsVpy9/wf4LnI6hLOYmqfiztiX7yBR0ck940SefvU3IFcCZfSrXwQVSZyvnDPPOeJqkUSo9SxlppJK/hi8ebt7bvb9yQiHxcEfqhI6ZqypGykEskyhXvL5emSBvRYpjzX77iEO8HiN8s7Jjm8SvlJJBzeJU3KaGAYVTVPhBRlAetZXjJ19QJIRSEV8OSFgmXNHRYl56mk683F1u7dNSJlRcJjmbAcGL8ML3BrxmuuV+F8oP8GFssMeaexKuOkauj6fd1wyyRhlTLH/yDLnCkwiLsXwZPJQYAFkqQ8VqwQKFJIwEpwu5oJMF9ACqaamuVkV5fliZN7ofawKTkERBas5oQVKUnyUgLnvVjumQpIVuZ5ed/5AStKUvE6K+sjXoiAB8DLgFT7mml3QR7pY8GOIpEh+T7nrDbS7YWSAVEMLguGI2DSXGpqWYEJyB1X95wXhhGHreii4JxF/giCl+RUguq0sx55XqZw5071MgQjPS0Wi5Rn4GnJge24VzaqauAGsknQm/y11iIIxiNtqDgDK90BcQgORvVLrRDrbuG/RfUafls+K9wY0HsaoH5BqegJUUv7w5v41e3rH2/e377y0RtZnezFiZsz8QcUBqYAq8EqSE42RgYdgiGGCbgNsIXo4elKFHCiW93xgtcMLb/iD6B6Y1nzsmU/+elvk0baGYYXFxfxTazdNkanvVjBzZr8C9ijMzY1X9nfjrdhBRc5CX4fHqsrDDTxMFjZrgd8RUacjp2C/FDIGBXr+WuryfC+FoqPCYN2x8KystbujviUTY0N7JXRC/6Egb/UyFpD4T07Gc3ov0YawZ8vvbbxeWtS7kFgq0CWTZ3wwAbAV4FUrFbRRZA2xuzR1UXAH3jSKB69hqBCfi4VsSYVZfRTWXAbMFOO0Qazt1f5IWipzE9gJX3zCq88I4G/1YxMgg6ZUizZazJjNFMeakhIhc4obVVgL775NtD3j3UdaYlt4Yl5cRJ1WWA8uF0zr7p9kC440+ZAF+ltifNyt+NpR3lkSXxf1gfIxfd9OnANSHs9Qp1sax5D4ZCQs4tWfJusNRdeg4L/2wigy6qrF73dUF7itq64ra9//Pnm/dWL+Lef3/7j9u3g2h33ZM/T+D/lXVDlzKgmKYtM7CJU7IqaB7kaFj7jgy5UDE0XajoZI0WIFUh69j3oDMoRf1Ce7z+LTHnt/BVKi+TkV5Y3/Lauy9qjFlpwyRVJRQaOJa/Ncw1RgVWkzFPSpTJ9bANMGCn4vd25OsE2LHbGR4wjhcdDKmoPFQ45P9KlkT8IqeLyYBzZUEMaBSVFHyk4vmqgsNrMCg54rmKbU+naXG+Y+DRSAFrjwDFCIwcfYqhJuYzRuggiAGpcjvYWpcK9P4NngD6VKBowiCmI+AxqvcbqplFMsm+KA9ZQCCYwmoIMUOxC8uvNLTjkI5TaHLOK4nUBpz6G5KeSoN/BfbuUjns1sIA6ApUSiFffvfknVHUmwP70ydS5NqiGmbCfqIwSjUJV/dhZHJiylCkW2YC2VcsbxPyqLWbUph/9b5eC3B/+olcHRmmICAm6UQSz0TBDjvOVTkijxV56Guz9C3ldc/6Bd0zI3aPiUtuhQFyTiw9atoDc/isAMoNxULM1+CrGbV1W4TDz1+UHXkT28h5tedOvR2KFsskyyPuQWXgNITVgU9ZiJ8C64FFZvGdyH5k0OLnbYJcB0GFSVo86kkfEgRHOH1ddy9q+fRZNDl9Pg/ttq7NkzwrImQQMicrBw6n/SSOZg0Yk6GQbOiKlWwhe7BjoGmLVSQi4WktM1xNJnxZDjmFTpVgOTQKIKCbJAn1RI6vIuXAn7++NCKtCx2hj04Hzarr9GxRZndlmfHrqz7auRL0aYwqpzUfToHIrXeTTtqbb3/7cEV9HG7pcHtnDEorHUnJIQimms6sL+Ge5dI3K0jQqAX2ply3KW6ZAn/IaY7puFWVfUlvne+qxgkzRjU0dRmnVo9qD3jtEMKeZjf0Deyq6TAClDQqsNyyaVm+btmfa+sFXjsPlejuUtEMAnqUJAD9EM1jCCBlA8U0Ouv50AvP83IXbK6DSzt7c1zrViSzWBTmOexnMRyuYq56D5UCAjIxtjJywBEbqW8v/v9xcjm7cOWU046fX3d+2og/N3kGDCOPWmzo/9uwOStMAXOIcA5cHwN8RLMXW36NvLwAN6ya/q92R7faBd6zNEzknMpOC7SfaIZO8B627gSfDxj96GSCvaAAlhrL3ElGvT6OTzq9bGCQkLJfOCaFwNXVhM+JiRrs9rOdN7TTpNv0BFFwM4ryl3HHlOdwFNcXxA6th/cx54XXEGwo2kRCbz6LLUX2fFJ5b2JXgyKPUSOpY5RyfNJACLj0twlM0PQOc5DoD4MnrGZec9sG0V1CwO4w6xO0ZNgGCb90nDREMUrskryGD6S61XuxcCfWifQ3fAxvzsodLkcA4t6XYUIOuYlsBt8+irvSMXn1OkxqfG0nMoMH2/AQgt8hEYmBkxiAFpT21Quixowxcl6G3wNlcAabdoYKpoYA4mXtp9tHtwA/RoFDp28HWJGubURpdXwTTEdr8AG0mRmlbldfn63VAFZOHWD1WCNdzvisHrQLL+6O4EX9Dltjh3N+tx+tWR1sR8DNLDojVtf/qFwYM2BZgPZvTKTTEd5AlBpiUrk3HThU0B9gstAvQQgF2UHGrTbM+x9iRojrkhAHk3F3Ddty+ePpDCN2Zdw7b6QJ1DtDCIax49Iwz6cg4QDic9IThEJxwxuB4h5AqjxJy0QxQvUlUw3KTH5wf2naUDg6zXa7JW3puC8dBtsDI67+7Y9CPxVJ80PF7OX6NyjPdgU4g1Heq0Tr8XEz+Urg7OYl5utIntsL3pLaAazZ/z87lhpn7uj+M8CzhQCe2MzCJxe/nmhZ5u2yje1VDPq7TK7NfT7j8P5GwfsAkNe43tJp6oOJL+0gzcZ5vqp5FZ/uRDf1Sac92SLPCWjgI1odzzWG2vqzaLApOrgEhJKl+b2KHiJ1aT5Ak0jiDuOFYVL/kjHYC2XJuFz7PFyNS1kmQShXsGMQlTjy9wYBzZhwcvvQDrztmMh0OLv2RjrsvQiEAU29Ds+xYcRzlLItSqlTgWHWJvDiaQK/jP2LQpoCkCIhZRoOMnsoc0nr0EcV+op8GdtBtrAEW0lzcgXw5aAGZ3+m1y7++OAxOAV3421mUPN+YdvAocIE+U1sD23fFLvLt73OCa5cxDhuNgrOzjh+g6uepnHn8cyf0Sq+F1H2cM3l7nlHF2SHWc1CW52WCqoj1OGbA8CwV9f9QD389+kg0NtUYOPOHhFeKfAcV/Fb/iRCJSaIdbv0ZG1soFWjiKKMfEWR4+skP4xg/LMXx05p81EtP9M9NJXBWN7pdbzTau0x7kTW2IcOBrhk6Cz1nXP8eKRYLSL/uTlFE4/gIIRbH1CipitwX4vCm3mlk9Qafai/lMqmFlieK47RMoP01W0KWpjGz1F7b5aIS9dwPgiNjCNHP9c6hHRhvLrfnWJqK1GMKHjIh0q10d6ytomkvznd12VSR2XlsEIDkjzGoO2+kOPFYv/Z6pJMjWjxqztGfmdv7XV18amv7CR87WK1GqBwlVHmAppyeu7gbKcxsCfY879nr3S9vEJ6/O2sVV/eWpmT2dHlO6TgRnr3pxZz6bVv7iesxUL0WFvdJr/0myEJzzX6gWmH1l65R/MY4eYxYCPbunGjq80PA0W7sfbqcGXX2MVt/02xnPWBuEuEYmPjT2Zr9RIRjUShZ09dQG87jnOtOMzqckHpjRrHbcwP13uih3Y1acB8pY1V6I236CButhOEYjRll6PQOlOGeP6RixyWo5llk5JnHYucx2W/a/L2BP15oheeTo5BHxNlzmEz7mxswshDzTneFYHSlwHxCskOjoGsdXSs/+cjaPpujh2M07bYmLbUj4yrk5j4uYeEbl4b6rTrgTeVpT0ubYyW99vOwuUTgOJs72A80sDj8RKPhspaj+482KCQsOJp25Mycuia3hCNHMBt0k6JeXoCr/w/cELTpiyQAAA=="
  },
  "electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_probe_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
single = importlib.util.module_from_spec(spec); spec.loader.exec_module(single)
sys.modules["electric_full_song"] = single
ROOT = single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = single.discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ditemukan. Gunakan VM aktif dari notebook 12 atau jalankan sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    single.model_python(ROOT)
spec = importlib.util.spec_from_file_location("drum_probe_embedded", TOOLS / "drum_probe.py")
drum = importlib.util.module_from_spec(spec); spec.loader.exec_module(drum)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
print("Runtime dipakai kembali:", ROOT)
print("Piano:", SOURCE_PATH)


## Satu generasi seluruh lagu

START=0 dan duration=None memakai seluruh piano asli (maksimum 600 detik). Tidak ada uji durasi atau chunk tambahan. Folder baru dibuat untuk setiap percobaan. Setelah error, jalankan sel download untuk mengambil report yang tersedia.


In [ ]:
from IPython.display import Audio, display
START = 0
DRUM_OUT = ROOT / "runs" / ("drum_full_" + uuid.uuid4().hex[:10])
report = drum.generate(ROOT, SOURCE_PATH, DRUM_OUT, start=START, duration=None, execute=not DRY_RUN)
print("Status:", report["status"], "|", DRUM_OUT)
if not DRY_RUN:
    for name in ("drums_preview.mp3", "mix_preview.mp3"):
        print(name)
        display(Audio(filename=str(DRUM_OUT / name), normalize=False))


## Simpan hasil penuh dan kirim feedback

Simpan drum_result.zip sebagai arsip WAV drum/mix. Upload drum_feedback.zip ke chat untuk menilai hasil seluruh lagu. Jika generasi gagal, feedback hanya berisi report yang tersedia.


In [ ]:
for name in ("drum_result.zip", "drum_feedback.zip"):
    path = DRUM_OUT / name
    if path.is_file():
        files.download(str(path))
if DRY_RUN:
    print("Plan saja; tidak ada audio model atau ZIP hasil yang dibuat")
